In [3]:
import tensorflow as tf 
import pandas as pd

PATH = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
PATH_test = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
COLUMNS = ['age','workclass','fnlwgt','education','education_num','marital','occupation','relationship','race','sex','capital_gain','capital_loss','hours_week','native_country','label']


In [4]:
df_train = pd.read_csv( PATH, skipinitialspace=True, names= COLUMNS, index_col=False)
df_test = pd.read_csv( PATH_test,skiprows=1, skipinitialspace=True, names= COLUMNS, index_col=False)

print(df_train.shape, df_test.shape)

print(df_train.dtypes)

(32561, 15) (16281, 15)
age               int64
workclass           str
fnlwgt            int64
education           str
education_num     int64
marital             str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_week        int64
native_country      str
label               str
dtype: object


In [5]:
label = {'<=50K': 0, '>50K': 1}
df_train.label = [label[item] for item in df_train.label]
label_t = {'<=50K.': 0, '>50K.': 1}
df_test.label = [label_t[item] for item in df_test.label]

In [ ]:
print(df_train["label"].value_counts())
print(df_test["label"].value_counts())

print(df_train.dtypes)

In [7]:
# Add features to the bucket
# Define continuous list

CONT_FEATURES = ['age', 'fnlwgt', 'capital_gain', 'education_num', 'capital_loss', 'hours_week']
# Define the categorial list
CATE_FEATURES = ['workclass', 'education', 'marital', 'occupation', 'relationship', 'race', 'sex', 'native_country']

from pandas.core.util.hashing import hash_pandas_object
# from sqlalchemy.orm import relationship
continuous_features = [tf.feature_column.numeric_column(k) for k in CONT_FEATURES]
relationship = tf.feature_column.categorical_column_with_vocabulary_list('relationship',['Husband','Not-in-family', 'Wife', 'Own-child', 'Unmarried','Other-relative'])
categorical_features = [tf.feature_column.categorical_column_with_hash_bucket(k,hash_bucket_size = 1000) for k in CATE_FEATURES]


Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.
Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.
Instructions for updating:
Use Keras preprocessing layers instead, either directly or via the `tf.keras.utils.FeatureSpace` utility. Each of `tf.feature_column.*` has a functional equivalent in `tf.keras.layers` for feature preprocessing when training a Keras model.


In [8]:
import os
model_dir = os.path.abspath('ongoing/train')
model = tf.estimator.LinearClassifier(n_classes=2, model_dir=model_dir, feature_columns=categorical_features + continuous_features)



Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.

INFO:tensorflow:Using default config.

INFO:tensorflow:Using config: {'_model_dir': 'e:\\LANGCHAIN\\Deeplearning\\ongoing\\train', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_i

In [12]:
import numpy as np

# Prepare data types required by TensorFlow Estimators
for col in CATE_FEATURES:
    df_train[col] = df_train[col].astype('object')
    df_test[col] = df_test[col].astype('object')

for col in CONT_FEATURES:
    df_train[col] = df_train[col].astype(np.float32)
    df_test[col] = df_test[col].astype(np.float32)

# Input function using tf.data.Dataset for high performance and compatibility
def get_input_fn(df, num_epochs=None, batch_size=128, shuffle=True):
    def input_fn():
        features = {col: df[col].to_numpy() for col in CONT_FEATURES + CATE_FEATURES}
        labels = df['label'].to_numpy(dtype=np.int32)
        dataset = tf.data.Dataset.from_tensor_slices((features, labels))
        if shuffle:
            dataset = dataset.shuffle(buffer_size=len(df))
        if num_epochs:
            dataset = dataset.repeat(num_epochs)
        else:
            dataset = dataset.repeat()
        dataset = dataset.batch(batch_size)
        return dataset
    return input_fn


In [13]:
# Train the LinearClassifier Estimator
model.train(input_fn=get_input_fn(df_train, num_epochs=None, batch_size=128, shuffle=True), steps=500)


Instructions for updating:
Use tf.keras instead.


INFO:tensorflow:Calling model_fn.



Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Done calling model_fn.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Create CheckpointSaverHook.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-1301
Instructions for updating:
Use standard file utilities to get mtimes.
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 1301...
IN

In [14]:
# Evaluate on the test dataset
results = model.evaluate(input_fn=get_input_fn(df_test, num_epochs=1, batch_size=128, shuffle=False))
print('Test Accuracy:', results['accuracy'])


INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.

INFO:tensorflow:Starting evaluation at 2026-09-25T19:00:15
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-1801
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Inference Time : 1.71891s
INFO:tensorflow:Finished evaluation at 2026-09-25-19:00:16
INFO:tensorflow:Saving dict for global step 1801: accuracy = 0.7874209, accuracy_baseline = 0.76377374, auc = 0.56024605, auc_precision_recall = 0.35900083, average_loss = 248.3023, global_step = 1801, label/mean = 0.23622628, loss = 247.32858, precision = 0.81609195, prediction/mean = 0.03737453, recall = 0.12922516
INFO:tensorflow:Saving 'checkpoint_path' summary for global step 1801: e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-1801
Test Accuracy: 0.7874209


In [23]:
import numpy as np

FEATURES = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_week', 'native_country']
LABEL = 'label'

def get_input_fn(data_set, num_epochs=None, n_batch=128, shuffle=True):
    # Cast string columns to numpy object and numeric to float32 to ensure compatibility with TensorFlow
    x_dict = {}
    for k in FEATURES:
        if k in CATE_FEATURES:
            x_dict[k] = data_set[k].astype(object)
        else:
            x_dict[k] = data_set[k].astype(np.float32)
    return tf.compat.v1.estimator.inputs.pandas_input_fn(
        x=pd.DataFrame(x_dict),
        y=pd.Series(data_set[LABEL].values, dtype=np.int32),
        batch_size=n_batch,
        num_epochs=num_epochs,
        shuffle=shuffle
    )


In [24]:
model.train(input_fn=get_input_fn(df_train,num_epochs=None, n_batch= 128, shuffle=False),steps = 1000)

Instructions for updating:
To construct input pipelines, use the `tf.data` module.
Instructions for updating:
To construct input pipelines, use the `tf.data` module.
INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Create CheckpointSaverHook.
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-1806
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
Instructions for updating:
To construct input pipelines, use the `tf.data` module.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 1806...
INFO:tensorflow:Saving checkpoints for 1806 into e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt.
INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 1806...
INFO:tensorflow:loss = 183.64444, step = 1806
INFO:tensorflow:global_step/sec: 415.03
INFO:tensorflow:loss = 135.07399, step = 1906 (0.242 sec)
INFO:tensorflow:

In [25]:
model.evaluate(input_fn=get_input_fn(df_test,num_epochs=1,n_batch=128,shuffle=False),steps=1000)

INFO:tensorflow:Calling model_fn.
INFO:tensorflow:Done calling model_fn.
INFO:tensorflow:Starting evaluation at 2026-09-25T19:15:15
INFO:tensorflow:Graph was finalized.
INFO:tensorflow:Restoring parameters from e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-2806
INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Evaluation [100/1000]
INFO:tensorflow:Inference Time : 0.52735s
INFO:tensorflow:Finished evaluation at 2026-09-25-19:15:15
INFO:tensorflow:Saving dict for global step 2806: accuracy = 0.76561636, accuracy_baseline = 0.76377374, auc = 0.6495759, auc_precision_recall = 0.39426756, average_loss = 26.701275, global_step = 2806, label/mean = 0.23622628, loss = 26.7006, precision = 0.50468165, prediction/mean = 0.1968036, recall = 0.42043683
INFO:tensorflow:Saving 'checkpoint_path' summary for global step 2806: e:\LANGCHAIN\Deeplearning\ongoing\train\model.ckpt-2806


{'accuracy': 0.76561636,
 'accuracy_baseline': 0.76377374,
 'auc': 0.6495759,
 'auc_precision_recall': 0.39426756,
 'average_loss': 26.701275,
 'label/mean': 0.23622628,
 'loss': 26.7006,
 'precision': 0.50468165,
 'prediction/mean': 0.1968036,
 'recall': 0.42043683,
 'global_step': 2806}

In [26]:
# salary is 0 at a very young age and then keeps increasing. Close to retirment it again decreases, hence we square age 
def square_var(df_t, df_te, var_name = 'age'):
    df_t['new'] = df_t[var_name].pow(2)
    df_te['new'] = df_te[var_name].pow(2)
    return df_t, df_te 

In [28]:
df_train_new, df_test_new = square_var(df_train , df_test , var_name = 'age')

In [30]:
print(df_train_new.shape, df_test_new.shape)

(32561, 16) (16281, 16)


In [31]:
CONTI_FEATURES_NEW = ['age','fnlwgt','capital_gain','education_num','capital_loss','horse_week','new']
continuous_features_new = [tf.feature_column.numeric_column(k) for k in CONTI_FEATURES_NEW]

In [32]:
model_1 = tf.estimator.LinearClassifier(
    model_dir = "ongoing/train1",
    feature_columns = categorical_features + continuous_features_new
)

INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': 'ongoing/train1', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


In [33]:
FEATURES_NEW = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_week', 'native_country']
def get_input_fn(data_set , num_epochs=None , n_batch=128 , shuffle=True):
    return tf.estimator.inputes.pandas_input_fn(
        x = pd.DataFrame({k: data_set[k].values for k in FEATURES_NEW}),
        y = pd.Series(data_set[LABEL].values),
        batch_size = n_batch,
        num_epochs = num_epochs,
        shuffle = shuffle
    )